# Natural forgetting: components and finite-update coupling

Full code is in the adjacent `.py` files. Training runs quietly in a background worker. Refresh **Status** manually. This precision study may require more than one session; Launch/resume preserves completed work. Read `README.md` for scientific definitions and limits.

In [ ]:
from pathlib import Path
import sys, json, importlib
ROOT = Path.cwd()
if not (ROOT / "run_study.py").exists():
    candidates = [ROOT / "natural_forgetting_components", ROOT / "outputs" / "natural_forgetting_components"]
    ROOT = next((p for p in candidates if (p / "run_study.py").exists()), ROOT)
if not (ROOT / "run_study.py").exists():
    raise FileNotFoundError("Open this notebook inside the extracted natural_forgetting_components folder.")
sys.path.insert(0, str(ROOT))
# Avoid importing a 'study' module left over from another experiment's folder.
for name in [p.stem for p in ROOT.glob("*.py")]:
    mod = sys.modules.get(name)
    if mod is not None and Path(getattr(mod, "__file__", "")).parent.resolve() != ROOT.resolve():
        del sys.modules[name]
import study
OUTPUT = ROOT / "runs" / "natural_forgetting_components_v1"
saved = OUTPUT / "settings.json"
SETTINGS = json.loads(saved.read_text()) if saved.exists() else study.defaults(OUTPUT)
# Changes to scientific settings belong before the FIRST launch.
# SETTINGS["seeds"] = [51]  # Optional one-seed pilot; keep all three for the full study.
SETTINGS["python"] = sys.executable
SETTINGS["hours"] = 23.0  # Per-session limit; launch again when paused.
print("Results:", SETTINGS["output"])
print("Python:", SETTINGS["python"])
print("Saved settings loaded." if saved.exists() else "Fresh configuration ready.")


## Environment check
This does not install packages or download models. Use your existing working GPU environment. If a dependency is absent, install it there from `requirements.txt`; do not replace your working PyTorch/CUDA installation.

In [ ]:
import torch, transformers, datasets, numpy, matplotlib
study.validate(SETTINGS)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable in this notebook kernel. Select the working GPU environment.")
print("GPU:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__, "Transformers:", transformers.__version__)
print("FP32 native training; eager attention; no automatic precision changes.")


## Launch / resume
Safe to rerun: an active worker is reused. No continuous output. Initial checkpoint/dataset downloads can take time.

In [ ]:
study.launch(SETTINGS)


## Status — rerun this cell to refresh
A recent heartbeat means the worker updated progress; it does not by itself prove the whole job is close to completion.

In [ ]:
print(json.dumps(study.status(SETTINGS), indent=2))


## Stop — run only when you want to pause
Wait until Status shows `alive: false`, then use Launch/resume. A partially computed path resumes from saved scalar records.

In [ ]:
# Uncomment to request a cooperative pause:
# print(study.stop(SETTINGS))


## Results
This refreshes the summaries from a consistent database snapshot and shows a plot. Completed and unresolved paths remain separate.

In [ ]:
if (Path(SETTINGS["output"]) / "results.sqlite").exists():
    analysis_dir = study.analyze(SETTINGS)
    study.show_results({"output": analysis_dir})
else:
    print("No saved measurements yet. Check Status.")


## Export
Creates a shareable ZIP without model checkpoints. Export refuses a folder with no results database. The individual JSON files can also be uploaded directly.

In [ ]:
from IPython.display import display, FileLink
zip_path = Path(study.export(SETTINGS))
print(zip_path)
try:
    display(FileLink(str(zip_path.relative_to(Path.cwd()))))
except ValueError:
    print("Open the printed folder in Jupyter's file browser to download the ZIP.")


## Last worker messages — only needed if status is failed

In [ ]:
log = Path(SETTINGS["output"]) / "worker.log"
print("\n".join(log.read_text(errors="replace").splitlines()[-40:]) if log.exists() else "No worker log yet.")


## Optional: genuinely new run
This preserves existing results and creates a new folder. Do not use it to resume. Wait for the current worker to stop first. If reopening the notebook later, set `OUTPUT` in Setup to the newly printed folder.

In [ ]:
# SETTINGS = study.restart(SETTINGS)
# print("NEW results folder:", SETTINGS["output"])
